# Running the experiments on Google Colab (A100)

This notebook reproduces the runs reported in the article: the seven classical models (default and tuned) and TabPFN v2 (Part A), and TabPFN-3.5 (Part B), all on the same partitions and on the same type of machine.

**Before starting**
1. On your computer, run `python exportar_para_colab.py --datos data/Base.csv` and copy the resulting `paquete_colab/` folder to `My Drive/paquete_colab` in Google Drive.
2. In Colab: *Runtime → Change runtime type → A100 GPU* (and high RAM).
3. For Part B, create a free Prior Labs account, accept the TabPFN-3.5 license, and store your API token as a Colab secret named `TABPFN_TOKEN` (key icon in the left sidebar).

Run the cells in order. Every script saves one row per fitted model in Drive and, if the session is interrupted, resumes where it stopped when the cell is run again (after re-running cells 1 and 2, or 1 and 7).

## 1. Mount Google Drive and check the machine

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc; grep -m1 'model name' /proc/cpuinfo
%cd /content/drive/MyDrive/paquete_colab

# Part A: classical models and TabPFN v2
## 2. Python 3.9 environment with the versions used in the article (about 3 minutes)
It is installed in `/content/venv39`, separate from Colab's Python. Re-run this cell if Colab disconnects.

In [ ]:
!pip -q install uv
!uv venv -q /content/venv39 --python 3.9
!uv pip install -q --python /content/venv39/bin/python --extra-index-url https://download.pytorch.org/whl/cu118 --index-strategy unsafe-best-match numpy==1.24.4 pandas==2.3.1 scipy==1.13.1 scikit-learn==1.6.1 joblib==1.5.1 threadpoolctl==3.6.0 xgboost==2.1.4 tensorflow==2.11.0 torch==2.7.1+cu118 tabpfn==2.1.3 huggingface-hub==0.36.1 einops==0.8.1
!/content/venv39/bin/python -c "import sys, importlib.metadata as m; print(sys.version.split()[0], {p: m.version(p) for p in ['numpy','pandas','scikit-learn','xgboost','tensorflow','torch','tabpfn']})"

## 3. Quick check (repetition 0, 1,000 rows, about 2 minutes)
Each line shows the new PR-AUC next to the published one; they should coincide.

In [ ]:
%cd /content/drive/MyDrive/paquete_colab
!/content/venv39/bin/python tiempos_colab.py --grupo principal --prueba-rapida

## 4. Full run (10 repetitions × 5 training sizes; several hours)
One complete repetition (all sizes and models) is run before the next one. Output: `resultados_tiempos_colab.csv`.

In [ ]:
%cd /content/drive/MyDrive/paquete_colab
!/content/venv39/bin/python tiempos_colab.py --grupo principal

## 5. Permutation importance: classical models and TabPFN v2 (about 20–30 minutes)
10,000 training rows, repetition 0, 5,000 test applications, 3 permutations per variable. Output: `importancia_colab/importancia.csv`.

In [ ]:
%cd /content/drive/MyDrive/paquete_colab
!/content/venv39/bin/python tiempos_colab.py --grupo importancia

# Part B: TabPFN-3.5
## 6. Install TabPFN 9.1.0 in Colab's Python and load the token

In [ ]:
!pip -q install tabpfn==9.1.0
import os
from google.colab import userdata
os.environ['TABPFN_TOKEN'] = userdata.get('TABPFN_TOKEN')
import importlib.metadata as m
print('tabpfn', m.version('tabpfn'), '| torch', m.version('torch'), '| token loaded:', bool(os.environ['TABPFN_TOKEN']))

## 7. Check data and partitions, then run TabPFN-3.5 (10 repetitions × 5 sizes)
`--solo-verificar` only checks that the data and every preprocessed test matrix match the fingerprints. Output: `resultados_tabpfn35.csv`.

In [ ]:
%cd /content/drive/MyDrive/paquete_colab
!python tabpfn35_colab.py --solo-verificar
!python tabpfn35_colab.py

## 8. Permutation importance of TabPFN-3.5 (about 10 minutes)

In [ ]:
%cd /content/drive/MyDrive/paquete_colab
!python tiempos_colab.py --grupo importancia35

## 9. Optional: timing control of TabPFN-3.5 in a second session
Re-runs repetition 0 of every size to check that inference times are stable across sessions (rows of group `v35` in `resultados_tiempos_colab.csv`; not used in the tables).

In [ ]:
%cd /content/drive/MyDrive/paquete_colab
!python tiempos_colab.py --grupo v35

## 10. When finished
Download from `My Drive/paquete_colab` and place in the `resultados_colab/` folder of the repository:
- `resultados_tiempos_colab.csv`
- `resultados_tabpfn35.csv`
- the `importancia_colab/` folder (contains `importancia.csv`)
- `entorno_tiempos_principal.json`, `entorno_colab.json` (TabPFN-3.5; renamed `entorno_tabpfn35.json` in the repository) and, if run, `entorno_tiempos_v35.json`

Then, on your computer, run `preparar_resultados_finales.py`, `figura_correlacion.py` and `generar_tablas_latex.py` (see the README).